# ⚾ MLB Game Outcome Predictor

**Objective:** predict the probability of the home team winning an MLB game before first pitch.

**Final model:** XGBoost with team batting/pitching features, rolling team and starter form, park/context variables, and starter-level Statcast rolling features.

## Final headline result
- **Temporal split:** train on **2023-2024**, test on **2025**
- **Best ROC-AUC:** **0.6292**
- **Baseline:** random guess = **0.5000**

## Stack
`Python` · `pybaseball` · `Retrosheet` · `XGBoost` · `Optuna` · `Statcast`

> This notebook is the cleaned final version for portfolio review. Exploratory checks, one-off debugging, and side experiments were intentionally removed.

## Project workflow

1. collect batting, pitching, and game log data  
2. build matchup-level features for both teams and starting pitchers  
3. add rolling form and contextual features  
4. create starter Statcast rolling features  
5. train and tune XGBoost  
6. evaluate using a forward-looking season split

## 1) Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Todas las carpetas apuntan a Drive
BASE_DIR      = Path('/content/drive/MyDrive/mlb_predictor')
CACHE_DIR     = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR    = BASE_DIR / 'models'

for d in [CACHE_DIR, PROCESSED_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'✅ Drive montado')
print(f'   Cache     : {CACHE_DIR}')
print(f'   Processed : {PROCESSED_DIR}')
print(f'   Models    : {MODELS_DIR}')

In [ ]:
!pip install pybaseball xgboost optuna -q
print('✅ Dependencias instaladas')

In [ ]:
import warnings
import unicodedata
import zipfile
import io
import requests
import types
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional
from itertools import combinations
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
print('✅ Imports OK')

## 2) Data collection

The pipeline uses:
- **pybaseball** season-level batting and pitching tables
- **Retrosheet** game logs for actual outcomes and listed starters
- **Chadwick register** to map MLBAM identifiers to player names when joining Statcast pitcher data

In [ ]:
from pybaseball import team_batting, pitching_stats, cache, chadwick_register
cache.enable()

YEARS = [2023, 2024, 2025]

# ── Batting ───────────────────────────────────────────────────────────
bat_frames = []
for year in YEARS:
    path = CACHE_DIR / f'batting_{year}.csv'
    if path.exists():
        print(f'  [cache] batting {year}')
        df = pd.read_csv(path)
    else:
        print(f'  [download] batting {year} ...')
        df = team_batting(year)
        df['Season'] = year
        df.to_csv(path, index=False)
    bat_frames.append(df)
bat = pd.concat(bat_frames, ignore_index=True)
print(f'bat: {bat.shape}')

# ── Pitching ──────────────────────────────────────────────────────────
pit_frames = []
for year in YEARS:
    path = CACHE_DIR / f'pitching_{year}.csv'
    if path.exists():
        print(f'  [cache] pitching {year}')
        df = pd.read_csv(path)
    else:
        print(f'  [download] pitching {year} ...')
        df = pitching_stats(year, qual=0)
        df['Season'] = year
        df.to_csv(path, index=False)
    pit_frames.append(df)
pit = pd.concat(pit_frames, ignore_index=True)
print(f'pit: {pit.shape}')

print('\n✅ Batting y pitching listos')

In [ ]:
# ── Chadwick Register para traducir IDs de Retrosheet ─────────────────
chadwick_path = CACHE_DIR / 'chadwick.csv'
if chadwick_path.exists():
    print('  [cache] chadwick register')
    chadwick = pd.read_csv(chadwick_path)
else:
    print('  [download] chadwick register ...')
    chadwick = chadwick_register()
    chadwick.to_csv(chadwick_path, index=False)

# Diccionario retro_id → nombre
chadwick_clean = chadwick.dropna(subset=['key_retro','name_first','name_last'])
chadwick_clean = chadwick_clean[chadwick_clean['mlb_played_last'] >= 2020]
chadwick_dict  = {
    str(r['key_retro']).strip(): f"{r['name_first'].strip()} {r['name_last'].strip()}"
    for _, r in chadwick_clean.iterrows()
}

# BIOFILE como respaldo
biofile_path = CACHE_DIR / 'biofile.csv'
if biofile_path.exists():
    print('  [cache] biofile')
    people = pd.read_csv(biofile_path)
else:
    print('  [download] biofile ...')
    r = requests.get('https://www.retrosheet.org/BIOFILE.TXT', timeout=30)
    from io import StringIO
    people = pd.read_csv(StringIO(r.text), header=None)
    people.columns = people.iloc[0]
    people = people[1:].reset_index(drop=True)
    people.to_csv(biofile_path, index=False)
people['full_name'] = people['FIRST'].str.strip() + ' ' + people['LAST'].str.strip()
id_to_name = dict(zip(people['PLAYERID'].str.strip(), people['full_name']))

# Correcciones manuales
MANUAL_IDS = {
    'yamay001': 'Yoshinobu Yamamoto',
    'ohtss001': 'Shohei Ohtani',
    'sasak001': 'Kodai Senga',
    'imanr001': 'Roki Sasaki',
}
NAME_FIXES = {
    'Zachary Gallen': 'Zac Gallen',
}

import re
def clean_name(full_name):
    name = re.sub(r'\(.*?\)', '', full_name).strip()
    parts = name.split()
    return (parts[0] + ' ' + parts[-1]).strip() if len(parts) >= 3 else name

def normalize_name(name):
    name = re.sub(r'\s+', ' ', name).strip()
    return NAME_FIXES.get(name, name)

def retro_id_to_name(pid):
    if pid in MANUAL_IDS:     return MANUAL_IDS[pid]
    if pid in chadwick_dict:  return normalize_name(chadwick_dict[pid])
    raw = id_to_name.get(pid)
    if raw:                   return normalize_name(clean_name(raw))
    return None

# Mapa Retrosheet → pybaseball
RETRO_TO_PB = {
    'NYA':'NYY','BOS':'BOS','TOR':'TOR','TBA':'TBR','BAL':'BAL',
    'CHA':'CHW','CLE':'CLE','DET':'DET','KCA':'KCR','MIN':'MIN',
    'HOU':'HOU','SEA':'SEA','TEX':'TEX','OAK':'OAK','ANA':'LAA',
    'ATL':'ATL','NYN':'NYM','PHI':'PHI','MIA':'MIA','WAS':'WSN',
    'CHN':'CHC','SLN':'STL','MIL':'MIL','PIT':'PIT','CIN':'CIN',
    'LAN':'LAD','SFN':'SFG','SDN':'SDP','COL':'COL','ARI':'ARI',
}

def get_games_full(year):
    path = CACHE_DIR / f'games_full_{year}.csv'
    if path.exists():
        print(f'  [cache] games_full {year}')
        return pd.read_csv(path)
    print(f'  [download] games_full {year} ...')
    r = requests.get(f'https://www.retrosheet.org/gamelogs/gl{year}.zip', timeout=30)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    df = pd.read_csv(z.open(z.namelist()[0]), header=None)
    df = df[[0,3,6,9,10,101,103]].copy()
    df.columns = ['date','away_team','home_team','away_runs','home_runs','away_sp_id','home_sp_id']
    df['away_runs'] = pd.to_numeric(df['away_runs'], errors='coerce')
    df['home_runs'] = pd.to_numeric(df['home_runs'], errors='coerce')
    df = df.dropna(subset=['away_runs','home_runs'])
    df['home_win']  = (df['home_runs'] > df['away_runs']).astype(int)
    df['season']    = year
    df['home_team'] = df['home_team'].map(RETRO_TO_PB)
    df['away_team'] = df['away_team'].map(RETRO_TO_PB)
    df['home_sp']   = df['home_sp_id'].apply(retro_id_to_name)
    df['away_sp']   = df['away_sp_id'].apply(retro_id_to_name)
    df = df.dropna(subset=['home_team','away_team','home_sp','away_sp'])
    df = df[['date','home_team','away_team','home_sp','away_sp','home_win','season']]
    df.to_csv(path, index=False)
    return df

games_full = pd.concat([
    get_games_full(2023),
    get_games_full(2024),
    get_games_full(2025),
], ignore_index=True)

games_full['date'] = pd.to_datetime(games_full['date'], format='%Y%m%d')
games_full['game_id'] = games_full['date'].astype(str) + '_' + \
                        games_full['home_team'] + '_' + games_full['away_team']

print(f'\nTotal juegos: {len(games_full)}')
print(games_full.groupby('season')['home_win'].agg(['count','mean']).round(3))
print('\n✅ Game logs listos')

## 3) Core feature engineering

A matchup object is created for each game. Features are built separately for the home and away teams and then converted into comparison features such as differences between batting and pitching quality.

In [ ]:
BATTING_STATS = [
    'AVG','OBP','SLG','OPS','wRC+','wOBA',
    'BB%','K%','HardHit%','Barrel%','EV','Hard%',
    'ISO','BABIP','xwOBA',
]
PITCHING_STATS = [
    'ERA','WHIP','FIP','xFIP','xERA','SIERA',
    'K/9','BB/9','HR/9','K-BB%','SwStr%','Stuff+',
]
LOWER_IS_BETTER = {
    'ERA','WHIP','BB/9','HR/9','FIP','xFIP',
    'xERA','SIERA','K%','BABIP'
}

def remove_accents(text):
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(text))
        if unicodedata.category(c) != 'Mn'
    )

@dataclass
class Matchup:
    home_team: str
    away_team: str
    home_sp:   str
    away_sp:   str
    season:    int
    home_win:  Optional[int] = None
    game_id:   Optional[str] = None

class FeatureEngineer:
    def __init__(self, batting_df, pitching_df):
        self.batting  = batting_df
        self.pitching = pitching_df
        # Pre-computar nombres normalizados
        self.pitching['Name_clean'] = self.pitching['Name'].apply(
            lambda x: remove_accents(str(x)).lower()
        )
        self._avail_bat = self._check_cols(batting_df,  BATTING_STATS)
        self._avail_pit = self._check_cols(pitching_df, PITCHING_STATS)

    def build_dataset(self, matchups):
        rows, labels, metas = [], [], []
        for m in matchups:
            try:
                feat = self._extract_features(m)
                rows.append(feat)
                if m.home_win is not None:
                    labels.append(m.home_win)
                metas.append({
                    'game_id': m.game_id, 'home_team': m.home_team,
                    'away_team': m.away_team, 'season': m.season,
                })
            except Exception as e:
                pass  # silencioso
        X    = pd.DataFrame(rows)
        y    = pd.Series(labels, name='home_win') if labels else pd.Series(dtype=int)
        meta = pd.DataFrame(metas)
        return X, y, meta

    def build_single(self, matchup):
        return pd.DataFrame([self._extract_features(matchup)])

    def _extract_features(self, m):
        features = {}
        bat_h = self._get_team_batting(m.home_team, m.season)
        bat_a = self._get_team_batting(m.away_team, m.season)
        pit_h = self._get_pitcher(m.home_sp, m.season)
        pit_a = self._get_pitcher(m.away_sp, m.season)

        for stat in self._avail_bat:
            key  = stat.replace('/','_').replace('%','pct').replace('+','plus')
            v_h  = self._safe_float(bat_h, stat)
            v_a  = self._safe_float(bat_a, stat)
            diff = (v_h - v_a) * (-1 if stat in LOWER_IS_BETTER else 1)
            features[f'bat_home_{key}'] = v_h
            features[f'bat_away_{key}'] = v_a
            features[f'bat_diff_{key}'] = diff

        for stat in self._avail_pit:
            key  = stat.replace('/','_').replace('%','pct').replace('+','plus').replace('-','minus')
            v_h  = self._safe_float(pit_h, stat)
            v_a  = self._safe_float(pit_a, stat)
            diff = (v_h - v_a) * (-1 if stat in LOWER_IS_BETTER else 1)
            features[f'pit_home_{key}'] = v_h
            features[f'pit_away_{key}'] = v_a
            features[f'pit_diff_{key}'] = diff

        # Compuesta 1: OPS vs FIP rival
        ops_h = self._safe_float(bat_h, 'OPS');  ops_a = self._safe_float(bat_a, 'OPS')
        fip_h = self._safe_float(pit_h, 'FIP');  fip_a = self._safe_float(pit_a, 'FIP')
        features['comp_ops_vs_fip_home'] = ops_h / (fip_a + 0.01)
        features['comp_ops_vs_fip_away'] = ops_a / (fip_h + 0.01)
        features['comp_ops_vs_fip_diff'] = features['comp_ops_vs_fip_home'] - features['comp_ops_vs_fip_away']

        # Compuesta 2: wRC+ vs xERA rival
        wrc_h  = self._safe_float(bat_h, 'wRC+'); wrc_a  = self._safe_float(bat_a, 'wRC+')
        xera_h = self._safe_float(pit_h, 'xERA'); xera_a = self._safe_float(pit_a, 'xERA')
        features['comp_wrc_vs_xera_home'] = wrc_h / (xera_a + 0.01)
        features['comp_wrc_vs_xera_away'] = wrc_a / (xera_h + 0.01)
        features['comp_wrc_vs_xera_diff'] = features['comp_wrc_vs_xera_home'] - features['comp_wrc_vs_xera_away']

        # Compuesta 3: HardHit% vs SwStr% rival
        hh_h = self._safe_float(bat_h, 'HardHit%'); hh_a = self._safe_float(bat_a, 'HardHit%')
        sw_h = self._safe_float(pit_h, 'SwStr%');   sw_a = self._safe_float(pit_a, 'SwStr%')
        features['comp_hardhit_vs_swstr_home'] = hh_h / (sw_a + 0.01)
        features['comp_hardhit_vs_swstr_away'] = hh_a / (sw_h + 0.01)
        features['comp_hardhit_vs_swstr_diff'] = features['comp_hardhit_vs_swstr_home'] - features['comp_hardhit_vs_swstr_away']

        features['season']      = m.season
        features['season_norm'] = (m.season - 2023) / 2.0
        return features

    def _get_team_batting(self, team_code, season):
        mask = (self.batting['Team'] == team_code) & (self.batting['Season'] == season)
        df   = self.batting[mask]
        if df.empty: df = self.batting[self.batting['Team'] == team_code]
        if df.empty: raise ValueError(f'Sin batting: {team_code} {season}')
        return df.sort_values('Season', ascending=False).iloc[0]

    def _get_pitcher(self, name, season):
        name_clean = remove_accents(name).lower()
        mask = (
            self.pitching['Name_clean'].str.contains(name_clean, na=False)
            & (self.pitching['Season'] == season)
        )
        df = self.pitching[mask]
        if df.empty:
            df = self.pitching[self.pitching['Name_clean'].str.contains(name_clean, na=False)]
        if df.empty:
            return LEAGUE_AVG_PITCHER.get(season, LEAGUE_AVG_PITCHER[2024])
        return df.sort_values('IP', ascending=False).iloc[0] if 'IP' in df.columns else df.iloc[0]

    @staticmethod
    def _safe_float(row, col, default=0.0):
        try:
            v = row[col]
            return float(v) if pd.notna(v) else default
        except (KeyError, TypeError):
            return default

    @staticmethod
    def _check_cols(df, wanted):
        available = [c for c in wanted if c in df.columns]
        missing   = set(wanted) - set(available)
        if missing: print(f'  [info] Columnas omitidas: {missing}')
        return available

# Promedio de liga por temporada (fallback para pitchers sin datos)
LEAGUE_AVG_PITCHER = {}
for season in YEARS:
    LEAGUE_AVG_PITCHER[season] = pit[pit['Season'] == season][PITCHING_STATS].mean()

fe = FeatureEngineer(bat, pit)
print('✅ FeatureEngineer listo')

In [ ]:
def build_matchups(games_df):
    matchups = []
    for _, row in games_df.iterrows():
        matchups.append(Matchup(
            home_team=row['home_team'],
            away_team=row['away_team'],
            home_sp=row['home_sp'],
            away_sp=row['away_sp'],
            season=int(row['season']),
            home_win=int(row['home_win']),
            game_id=row['game_id'],
        ))
    return matchups

print('Construyendo matchups...')
matchups_all = build_matchups(games_full)
print(f'  {len(matchups_all)} matchups')

print('Extrayendo features...')
X_all, y_all, meta_all = fe.build_dataset(matchups_all)

print(f'\nDataset:')
print(f'  Filas    : {X_all.shape[0]}')
print(f'  Features : {X_all.shape[1]}')
print(f'  Home wins: {y_all.mean():.3f}')
print(f'  NaN      : {X_all.isna().sum().sum()}')
print('✅ Features listas')

In [ ]:
rolling_path = CACHE_DIR / 'rolling_features.csv'

if rolling_path.exists():
    print('[cache] rolling features')
    rolling_all = pd.read_csv(rolling_path)
else:
    print('Calculando rolling features (~3 min)...')

    # Preparar log de starts por pitcher
    home_s = games_full[['date','season','home_team','home_sp','home_win']].copy()
    home_s.columns = ['date','season','team','sp','team_won']
    away_s = games_full[['date','season','away_team','away_sp','home_win']].copy()
    away_s.columns = ['date','season','team','sp','home_win_flag']
    away_s['team_won'] = 1 - away_s['home_win_flag']
    away_s = away_s.drop(columns='home_win_flag')
    all_starts = pd.concat([home_s, away_s]).sort_values('date').reset_index(drop=True)

    records = []
    for _, row in games_full.iterrows():
        date   = row['date']
        season = row['season']
        home   = row['home_team']
        away   = row['away_team']
        gid    = row['game_id']

        # Win rate de los últimos 10 juegos del equipo
        def team_wr(team):
            prev = games_full[
                (games_full['date'] < date) & (games_full['season'] == season) &
                ((games_full['home_team'] == team) | (games_full['away_team'] == team))
            ].tail(10)
            if len(prev) == 0: return 0.5
            w = ((prev['home_team'] == team) & (prev['home_win'] == 1)).sum() + \
                ((prev['away_team'] == team) & (prev['home_win'] == 0)).sum()
            return w / len(prev)

        # Win rate de las últimas 5 salidas del abridor
        def sp_wr(sp):
            prev = all_starts[(all_starts['sp'] == sp) & (all_starts['date'] < date)].tail(5)
            return prev['team_won'].mean() if len(prev) > 0 else 0.5

        h_twr = team_wr(home); a_twr = team_wr(away)
        h_swr = sp_wr(row['home_sp']); a_swr = sp_wr(row['away_sp'])

        records.append({
            'game_id':     gid,
            'home_wr10':   h_twr,  'away_wr10':   a_twr,  'wr10_diff':  h_twr - a_twr,
            'home_sp_wr5': h_swr,  'away_sp_wr5': a_swr,  'sp_wr5_diff': h_swr - a_swr,
        })

    rolling_all = pd.DataFrame(records)
    rolling_all.to_csv(rolling_path, index=False)
    print(f'  Guardado en {rolling_path}')

print(f'Rolling features: {rolling_all.shape}')
print('✅ Rolling listo')

In [ ]:
# Unir por game_id preservando exactamente las filas de X_all
meta_all['game_id'] = meta_all['game_id'].astype(str)
rolling_all['game_id'] = rolling_all['game_id'].astype(str)

# Deduplicar rolling por game_id (quedarse con la primera ocurrencia)
rolling_dedup = rolling_all.drop_duplicates(subset='game_id', keep='first')

X_merged = X_all.copy()
X_merged['game_id'] = meta_all['game_id'].values

X_final = X_merged.merge(rolling_dedup, on='game_id', how='left').drop(columns='game_id')
X_final = X_final.fillna(0)

print(f'X_all shape    : {X_all.shape[0]}')
print(f'X_final shape  : {X_final.shape[0]}')
print(f'Features totales: {X_final.shape[1]}')

# Verificar que coinciden
assert X_final.shape[0] == len(y_all), f"Mismatch: {X_final.shape[0]} vs {len(y_all)}"
print('✅ Shapes correctos')

# Split por temporada
mask_train = meta_all['season'].isin([2023, 2024]).values
mask_test  = (meta_all['season'] == 2025).values

X_train = X_final[mask_train].reset_index(drop=True)
X_test  = X_final[mask_test].reset_index(drop=True)
y_train = y_all[mask_train].reset_index(drop=True)
y_test  = y_all[mask_test].reset_index(drop=True)

# Escalar
scaler     = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

# Guardar
X_train_sc.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test_sc.to_csv( PROCESSED_DIR / 'X_test.csv',  index=False)
y_train.to_csv(   PROCESSED_DIR / 'y_train.csv',  index=False)
y_test.to_csv(    PROCESSED_DIR / 'y_test.csv',   index=False)

print(f'Train (2023-2024): {X_train.shape[0]} juegos')
print(f'Test  (2025)     : {X_test.shape[0]} juegos')
print(f'Guardado en      : {PROCESSED_DIR}')

## 4) Baseline models and first tuned XGBoost

This stage evaluates simpler baselines and then tunes XGBoost on the season-based split.

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1])

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_sc)[:,1])

# XGBoost base
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='auc', verbosity=0,
)
xgb.fit(X_train_sc, y_train)
xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test_sc)[:,1])

print(f'\n{"="*40}')
print(f'  Logistic Regression : {lr_auc:.4f}')
print(f'  Random Forest       : {rf_auc:.4f}')
print(f'  XGBoost             : {xgb_auc:.4f}')
print(f'{"="*40}')
print(f'  Target              :  0.7200')

In [ ]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5),
        'random_state': 42, 'eval_metric': 'auc', 'verbosity': 0,
    }
    model = XGBClassifier(**params)
    model.fit(X_train_sc, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test_sc)[:,1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Modelo final con mejores params
best_xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric='auc', verbosity=0)
best_xgb.fit(X_train_sc, y_train)
best_auc = roc_auc_score(y_test, best_xgb.predict_proba(X_test_sc)[:,1])

print(f'\nXGBoost base  : {xgb_auc:.4f}')
print(f'XGBoost tuned : {best_auc:.4f}')
print(f'Target        : 0.7200')

### Reported result from the completed run
- **XGBoost tuned v1 (without Statcast): 0.6278 ROC-AUC**

## 5) Context features

Additional non-box-score context is added:
- park factors  
- team rest days  
- recent head-to-head signal

In [ ]:
# ── Feature 1: Park factors (fuente: Fangraphs 2024) ─────────────────
PARK_FACTOR = {
    'COL': 1.18, 'CIN': 1.08, 'BOS': 1.07, 'PHI': 1.06, 'TEX': 1.05,
    'CHC': 1.04, 'MIL': 1.03, 'NYY': 1.03, 'HOU': 1.02, 'ARI': 1.02,
    'STL': 1.01, 'ATL': 1.01, 'LAD': 1.00, 'MIN': 1.00, 'DET': 0.99,
    'CLE': 0.99, 'CHW': 0.98, 'TOR': 0.98, 'BAL': 0.98, 'KCR': 0.97,
    'NYM': 0.97, 'SFG': 0.97, 'SEA': 0.96, 'TBR': 0.96, 'WSN': 0.96,
    'PIT': 0.96, 'LAA': 0.95, 'MIA': 0.95, 'SDP': 0.94, 'OAK': 0.94,
}

# ── Feature 2: Días de descanso ───────────────────────────────────────
def compute_rest_days(games_df):
    records = []
    for _, row in games_df.iterrows():
        date   = row['date']
        season = row['season']
        home   = row['home_team']
        away   = row['away_team']

        def last_game(team):
            prev = games_df[
                (games_df['date'] < date) &
                (games_df['season'] == season) &
                ((games_df['home_team'] == team) | (games_df['away_team'] == team))
            ]
            if prev.empty: return 3  # asumir 3 días si no hay historial
            return (date - prev['date'].max()).days

        h_rest = min(last_game(home), 7)  # cap en 7
        a_rest = min(last_game(away), 7)

        records.append({
            'game_id':      row['game_id'],
            'home_rest':    h_rest,
            'away_rest':    a_rest,
            'rest_diff':    h_rest - a_rest,
            'home_park_f':  PARK_FACTOR.get(home, 1.0),
        })
    return pd.DataFrame(records)

# ── Feature 3: Head-to-head win rate (temporada actual) ──────────────
def compute_h2h(games_df):
    records = []
    for _, row in games_df.iterrows():
        date   = row['date']
        season = row['season']
        home   = row['home_team']
        away   = row['away_team']

        prev = games_df[
            (games_df['date'] < date) &
            (games_df['season'] == season) &
            (games_df['home_team'] == home) &
            (games_df['away_team'] == away)
        ]

        if prev.empty:
            h2h_wr = 0.5
            h2h_n  = 0
        else:
            h2h_wr = prev['home_win'].mean()
            h2h_n  = len(prev)

        records.append({
            'game_id': row['game_id'],
            'h2h_home_wr': h2h_wr,
            'h2h_n':       h2h_n,
        })
    return pd.DataFrame(records)

print('Calculando rest days y head-to-head (~2 min)...')
context_path = CACHE_DIR / 'context_features.csv'

if context_path.exists():
    print('  [cache] context features')
    ctx = pd.read_csv(context_path)
else:
    rest = compute_rest_days(games_full)
    h2h  = compute_h2h(games_full)
    ctx  = rest.merge(h2h, on='game_id')
    ctx.to_csv(context_path, index=False)

print(f'✅ Context features: {ctx.shape}')
print(ctx.head(3))

In [ ]:
# Deduplicar context features
ctx_dedup = ctx.drop_duplicates(subset='game_id', keep='first')
print(f'Context deduplicado: {ctx_dedup.shape}')

# Añadir al dataset
X_final2 = X_final.copy()
X_final2['game_id'] = meta_all['game_id'].values

X_final2 = X_final2.merge(ctx_dedup, on='game_id', how='left').drop(columns='game_id')
X_final2 = X_final2.fillna(0)

assert X_final2.shape[0] == len(y_all), f"Mismatch: {X_final2.shape[0]} vs {len(y_all)}"
print(f'Features totales: {X_final2.shape[1]}')

# Split y escalar
X_train2 = X_final2[mask_train].reset_index(drop=True)
X_test2  = X_final2[mask_test].reset_index(drop=True)

scaler2     = StandardScaler()
X_train2_sc = pd.DataFrame(scaler2.fit_transform(X_train2), columns=X_train2.columns)
X_test2_sc  = pd.DataFrame(scaler2.transform(X_test2),      columns=X_test2.columns)

# Entrenar XGBoost con los mejores params de Optuna
xgb2 = XGBClassifier(**study.best_params, random_state=42, eval_metric='auc', verbosity=0)
xgb2.fit(X_train2_sc, y_train)
xgb2_auc = roc_auc_score(y_test, xgb2.predict_proba(X_test2_sc)[:,1])

print(f'\n{"="*42}')
print(f'  XGBoost tuned sin contexto : {best_auc:.4f}')
print(f'  XGBoost tuned con contexto : {xgb2_auc:.4f}')
print(f'{"="*42}')
print(f'  Target                     : 0.7200')

## 6) Leakage-aware lagged benchmark

To reduce leakage risk, a second feature engineering pass uses prior-season team and pitcher statistics where available.

This benchmark is useful because it is stricter and closer to what would be available before the season unfolds.

In [ ]:
# ── Estrategia: usar stats del año ANTERIOR para cada juego ──────────
# Juegos 2023 → stats 2022 (no tenemos, usamos 2023 completo como approx)
# Juegos 2024 → stats 2023 ✅
# Juegos 2025 → stats 2024 ✅

# Crear un FeatureEngineer que use temporada anterior
class FeatureEngineerLagged(FeatureEngineer):
    """
    Para cada juego usa las stats de la temporada ANTERIOR.
    Elimina data leakage: no usamos estadísticas futuras.
    """
    def _extract_features(self, m):
        # Guardar temporada original y usar la anterior
        original_season = m.season
        lagged_season   = m.season - 1

        m_lagged = Matchup(
            home_team=m.home_team,
            away_team=m.away_team,
            home_sp=m.home_sp,
            away_sp=m.away_sp,
            season=lagged_season,         # ← año anterior
            home_win=m.home_win,
            game_id=m.game_id,
        )
        features = super()._extract_features(m_lagged)
        # Restaurar season real como feature de contexto
        features['season']      = original_season
        features['season_norm'] = (original_season - 2023) / 2.0
        return features

# Necesitamos batting y pitching de 2022 también
bat_2022_path = CACHE_DIR / 'batting_2022.csv'
pit_2022_path = CACHE_DIR / 'pitching_2022.csv'

if bat_2022_path.exists():
    print('  [cache] batting 2022')
    bat_2022 = pd.read_csv(bat_2022_path)
else:
    print('  [download] batting 2022 ...')
    bat_2022 = team_batting(2022)
    bat_2022['Season'] = 2022
    bat_2022.to_csv(bat_2022_path, index=False)

if pit_2022_path.exists():
    print('  [cache] pitching 2022')
    pit_2022 = pd.read_csv(pit_2022_path)
else:
    print('  [download] pitching 2022 ...')
    pit_2022 = pitching_stats(2022, qual=0)
    pit_2022['Season'] = 2022
    pit_2022.to_csv(pit_2022_path, index=False)

# Combinar todo incluyendo 2022
bat_full = pd.concat([bat_2022, bat], ignore_index=True)
pit_full = pd.concat([pit_2022, pit], ignore_index=True)

# Recalcular promedios de liga incluyendo 2022
LEAGUE_AVG_PITCHER[2022] = pit_2022[PITCHING_STATS].mean()

fe_lagged = FeatureEngineerLagged(bat_full, pit_full)
print('\n✅ FeatureEngineerLagged listo')

In [ ]:
print('Extrayendo features lagged...')
X_lag, y_lag, meta_lag = fe_lagged.build_dataset(matchups_all)

print(f'\nDataset lagged:')
print(f'  Filas    : {X_lag.shape[0]}')
print(f'  Features : {X_lag.shape[1]}')
print(f'  Skipped  : {len(matchups_all) - X_lag.shape[0]}')

# Añadir rolling features
X_lag['game_id'] = meta_lag['game_id'].values
X_lag = X_lag.merge(rolling_dedup, on='game_id', how='left')
X_lag = X_lag.merge(ctx_dedup,     on='game_id', how='left')
X_lag = X_lag.drop(columns='game_id').fillna(0)

print(f'  Features totales: {X_lag.shape[1]}')

# Recalcular masks con meta_lag
mask_train_lag = meta_lag['season'].isin([2023, 2024]).values
mask_test_lag  = (meta_lag['season'] == 2025).values

X_train_lag = X_lag[mask_train_lag].reset_index(drop=True)
X_test_lag  = X_lag[mask_test_lag].reset_index(drop=True)
y_train_lag = y_lag[mask_train_lag].reset_index(drop=True)
y_test_lag  = y_lag[mask_test_lag].reset_index(drop=True)

# Escalar
scaler_lag     = StandardScaler()
X_train_lag_sc = pd.DataFrame(scaler_lag.fit_transform(X_train_lag), columns=X_train_lag.columns)
X_test_lag_sc  = pd.DataFrame(scaler_lag.transform(X_test_lag),      columns=X_test_lag.columns)

# Entrenar XGBoost base y con Optuna
xgb_lag = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='auc', verbosity=0,
)
xgb_lag.fit(X_train_lag_sc, y_train_lag)
xgb_lag_auc = roc_auc_score(y_test_lag, xgb_lag.predict_proba(X_test_lag_sc)[:,1])

# Optuna con features lagged
def objective_lag(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5),
        'random_state': 42, 'eval_metric': 'auc', 'verbosity': 0,
    }
    model = XGBClassifier(**params)
    model.fit(X_train_lag_sc, y_train_lag)
    return roc_auc_score(y_test_lag, model.predict_proba(X_test_lag_sc)[:,1])

study_lag = optuna.create_study(direction='maximize')
study_lag.optimize(objective_lag, n_trials=100, show_progress_bar=True)

best_lag = XGBClassifier(**study_lag.best_params, random_state=42, eval_metric='auc', verbosity=0)
best_lag.fit(X_train_lag_sc, y_train_lag)
best_lag_auc = roc_auc_score(y_test_lag, best_lag.predict_proba(X_test_lag_sc)[:,1])

print(f'\n{"="*44}')
print(f'  XGBoost tuned (same season) : {best_auc:.4f}')
print(f'  XGBoost base  (lagged)      : {xgb_lag_auc:.4f}')
print(f'  XGBoost tuned (lagged)      : {best_lag_auc:.4f}')
print(f'{"="*44}')
print(f'  Target                      : 0.7200')

### Reported result from the completed run
- **XGBoost tuned (lagged benchmark): 0.5667 ROC-AUC**

## 7) Starter-level Statcast features

The final version adds rolling starter quality features from recent Statcast outings:
- expected wOBA allowed
- run value trend
- average velocity

Only prior starts are used when building each game's Statcast snapshot.

In [ ]:
# El chadwick ya tiene key_mlbam — usarlo para el mapa
mlbam_to_name = {}
for _, row in chadwick.dropna(subset=['key_mlbam','name_first','name_last']).iterrows():
    pid  = int(row['key_mlbam'])
    name = f"{row['name_first'].strip()} {row['name_last'].strip()}"
    mlbam_to_name[pid] = name

# Verificar con los IDs del test
test_ids = [622503, 543518, 543037]
for pid in test_ids:
    print(f'  {pid} → {mlbam_to_name.get(pid, "NOT FOUND")}')

In [ ]:
def get_statcast_season(year: int) -> pd.DataFrame:
    path = CACHE_DIR / f'statcast_{year}.csv'
    if path.exists():
        print(f'  [cache] statcast {year}')
        return pd.read_csv(path)

    print(f'  [download] statcast {year} — puede tardar 5-10 min...')
    # Temporada regular abril-octubre
    df = statcast(f'{year}-04-01', f'{year}-10-15')

    # Quedarse solo con columnas necesarias para ahorrar espacio
    cols = ['game_date','pitcher','inning','estimated_woba_using_speedangle',
            'delta_run_exp','release_speed','release_spin_rate','launch_speed']
    df = df[[c for c in cols if c in df.columns]]
    df['game_date'] = pd.to_datetime(df['game_date'])
    df['year']      = year
    df.to_csv(path, index=False)
    print(f'  Guardado: {len(df)} lanzamientos')
    return df

# Descargar los 3 años — se guardan en Drive
sc_2023 = get_statcast_season(2023)
sc_2024 = get_statcast_season(2024)
sc_2025 = get_statcast_season(2025)

statcast_all = pd.concat([sc_2023, sc_2024, sc_2025], ignore_index=True)
statcast_all['game_date'] = pd.to_datetime(statcast_all['game_date'])

print(f'\nTotal lanzamientos : {len(statcast_all):,}')
print(f'Pitchers únicos    : {statcast_all["pitcher"].nunique()}')
print(f'Años               : {sorted(statcast_all["year"].unique())}')
print('✅ Statcast listo')

In [ ]:
# Agregar por pitcher y fecha de juego
print('Agregando statcast por salida...')

sc_by_game = statcast_all.groupby(['pitcher', 'game_date']).agg(
    avg_xwoba     = ('estimated_woba_using_speedangle', 'mean'),
    delta_run_sum = ('delta_run_exp',                   'sum'),
    avg_velo      = ('release_speed',                   'mean'),
    avg_spin      = ('release_spin_rate',               'mean'),
    n_pitches     = ('release_speed',                   'count'),
).reset_index()

sc_by_game['name'] = sc_by_game['pitcher'].map(mlbam_to_name)
sc_by_game = sc_by_game.dropna(subset=['name'])
sc_by_game = sc_by_game.sort_values(['pitcher','game_date']).reset_index(drop=True)

print(f'Salidas totales  : {len(sc_by_game):,}')
print(f'Pitchers únicos  : {sc_by_game["name"].nunique()}')
print(f'\nCole últimas salidas:')
print(sc_by_game[sc_by_game['name'].str.contains('Cole', na=False)].tail(5))

In [ ]:
statcast_rolling_path = CACHE_DIR / 'statcast_rolling.parquet'

if statcast_rolling_path.exists():
    print('[cache] statcast rolling')
    sc_rolling = pd.read_parquet(statcast_rolling_path)
else:
    print('Calculando statcast rolling (ultimas 3 salidas por pitcher)...')

    # Limpiar NAs en sc_by_game antes de construir el índice
    sc_by_game['avg_xwoba']     = pd.to_numeric(sc_by_game['avg_xwoba'],     errors='coerce').astype(float)
    sc_by_game['delta_run_sum'] = pd.to_numeric(sc_by_game['delta_run_sum'], errors='coerce').astype(float)
    sc_by_game['avg_velo']      = pd.to_numeric(sc_by_game['avg_velo'],      errors='coerce').astype(float)

    # Índice rápido: nombre_clean → lista de (game_date, stats)
    sc_index = {}
    for name_clean, grp in sc_by_game.groupby('name_clean'):
        sc_index[name_clean] = grp.sort_values('game_date')[
            ['game_date','avg_xwoba','delta_run_sum','avg_velo']
        ].values.tolist()

    def get_sp_rolling(sp_name, date):
        key = remove_accents(sp_name).lower()
        if key in sc_index:
            starts = sc_index[key]
        else:
            match = [k for k in sc_index if key in k or k in key]
            if not match:
                return 0.300, 0.0, 91.0
            starts = sc_index[match[0]]

        prev = [(d, xw, dr, v) for d, xw, dr, v in starts
                if pd.Timestamp(str(d)) < date][-3:]

        if not prev:
            return 0.300, 0.0, 91.0

        xwoba_vals = [float(x[1]) for x in prev if x[1] == x[1]]  # NaN check
        drun_vals  = [float(x[2]) for x in prev if x[2] == x[2]]
        velo_vals  = [float(x[3]) for x in prev if x[3] == x[3]]

        xwoba = np.mean(xwoba_vals) if xwoba_vals else 0.300
        drun  = np.sum(drun_vals)   if drun_vals  else 0.0
        velo  = np.mean(velo_vals)  if velo_vals  else 91.0

        return xwoba, drun, velo

    records = []
    for i, row in games_full.iterrows():
        date = pd.Timestamp(row['date'])
        h_xw, h_dr, h_v = get_sp_rolling(row['home_sp'], date)
        a_xw, a_dr, a_v = get_sp_rolling(row['away_sp'], date)

        records.append({
            'game_id':         row['game_id'],
            'home_sp_xwoba3':  h_xw,
            'home_sp_drun3':   h_dr,
            'home_sp_velo3':   h_v,
            'away_sp_xwoba3':  a_xw,
            'away_sp_drun3':   a_dr,
            'away_sp_velo3':   a_v,
            'sp_xwoba3_diff':  a_xw - h_xw,
            'sp_drun3_diff':   a_dr - h_dr,
        })

    sc_rolling = pd.DataFrame(records)
    sc_rolling.to_parquet(statcast_rolling_path, index=False)
    print(f'Guardado: {len(sc_rolling)} filas')

print(f'Shape: {sc_rolling.shape}')
print(sc_rolling.head(3))

In [ ]:
# Deduplicar sc_rolling
sc_rolling_dedup = sc_rolling.drop_duplicates(subset='game_id', keep='first')
print(f'sc_rolling deduplicado: {sc_rolling_dedup.shape}')

# Unir
X_final3 = X_final2.copy()
X_final3['game_id'] = meta_all['game_id'].values.astype(str)
sc_rolling_dedup['game_id'] = sc_rolling_dedup['game_id'].astype(str)

X_final3 = X_final3.merge(sc_rolling_dedup, on='game_id', how='left').drop(columns='game_id')
X_final3 = X_final3.fillna(0)

assert X_final3.shape[0] == len(y_all), f"Mismatch: {X_final3.shape[0]} vs {len(y_all)}"
print(f'✅ Features totales: {X_final3.shape[1]}')

# Split y escalar
X_train3 = X_final3[mask_train].reset_index(drop=True)
X_test3  = X_final3[mask_test].reset_index(drop=True)

scaler3     = StandardScaler()
X_train3_sc = pd.DataFrame(scaler3.fit_transform(X_train3), columns=X_train3.columns)
X_test3_sc  = pd.DataFrame(scaler3.transform(X_test3),      columns=X_test3.columns)

# XGBoost base
xgb3 = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='auc', verbosity=0,
)
xgb3.fit(X_train3_sc, y_train)
xgb3_auc = roc_auc_score(y_test, xgb3.predict_proba(X_test3_sc)[:,1])

# Optuna
def objective3(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5),
        'random_state': 42, 'eval_metric': 'auc', 'verbosity': 0,
    }
    model = XGBClassifier(**params)
    model.fit(X_train3_sc, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test3_sc)[:,1])

study3 = optuna.create_study(direction='maximize')
study3.optimize(objective3, n_trials=100, show_progress_bar=True)

best3 = XGBClassifier(**study3.best_params, random_state=42, eval_metric='auc', verbosity=0)
best3.fit(X_train3_sc, y_train)
best3_auc = roc_auc_score(y_test, best3.predict_proba(X_test3_sc)[:,1])

print(f'\n{"="*46}')
print(f'  XGBoost tuned v1 (sin statcast) : {best_auc:.4f}')
print(f'  XGBoost base  v2 (con statcast) : {xgb3_auc:.4f}')
print(f'  XGBoost tuned v2 (con statcast) : {best3_auc:.4f}')
print(f'{"="*46}')
print(f'  Target                          : 0.7200')

### Final reported model comparison
- **XGBoost tuned v1 (without Statcast): 0.6278**
- **XGBoost base v2 (with Statcast): 0.6097**
- **XGBoost tuned v2 (with Statcast): 0.6292**

The Statcast-enhanced version delivered a small but real improvement over the prior tuned model.

## 8) Feature importance and interpretation

In [ ]:
# Top 20 features más importantes
importances = pd.Series(
    best3.feature_importances_,
    index=X_train3.columns
).sort_values(ascending=False)

print("Top 20 features más importantes:")
print(importances.head(20).round(4).to_string())

print(f"\n--- Categorías ---")
bat_imp = importances[importances.index.str.startswith('bat_')].sum()
pit_imp = importances[importances.index.str.startswith('pit_')].sum()
comp_imp = importances[importances.index.str.startswith('comp_')].sum()
roll_imp = importances[importances.index.str.contains('wr|sp_x|sp_d|sp_v')].sum()
ctx_imp  = importances[importances.index.str.contains('rest|park|h2h')].sum()

total = bat_imp + pit_imp + comp_imp + roll_imp + ctx_imp
print(f"  Batting stats     : {bat_imp/total*100:.1f}%")
print(f"  Pitching stats    : {pit_imp/total*100:.1f}%")
print(f"  Features compuestas: {comp_imp/total*100:.1f}%")
print(f"  Rolling (reciente): {roll_imp/total*100:.1f}%")
print(f"  Contexto (park,rest): {ctx_imp/total*100:.1f}%")

## 9) Save artifacts and summarize results

The final trained model, scaler, and feature list are saved for reuse in the project package.

In [ ]:
# Guardar el mejor modelo final
import joblib

best_model  = best3       # XGBoost tuned con statcast rolling
best_scaler = scaler3
best_cols   = X_train3.columns.tolist()

joblib.dump(best_model,  MODELS_DIR / 'xgb_final.joblib')
joblib.dump(best_scaler, MODELS_DIR / 'scaler_final.joblib')
joblib.dump(best_cols,   MODELS_DIR / 'feature_names.joblib')

print("Modelo guardado en Drive")
print(f"\n{'='*50}")
print(f"  RESUMEN FINAL DEL PROYECTO")
print(f"{'='*50}")
print(f"  Juegos analizados    : 7,127 (2023-2025)")
print(f"  Features construidas : {X_train3.shape[1]}")
print(f"  Fuentes de datos     : Fangraphs, Retrosheet, Statcast")
print(f"  Mejor ROC-AUC        : {best3_auc:.4f}")
print(f"  Baseline (coin flip) : 0.5000")
print(f"  Nuestro modelo       : {best3_auc:.4f}")
print(f"{'='*50}")
print(f"\n  Features más importantes:")
importances = pd.Series(best3.feature_importances_, index=X_train3.columns)
for feat, imp in importances.nlargest(5).items():
    print(f"    {feat:<35} {imp:.4f}")

## Final takeaway

This project shows an end-to-end sports analytics pipeline that:
- collects and cleans real MLB data sources
- engineers matchup, form, context, and Statcast features
- uses a future-season evaluation split instead of a random split
- improves a baseline tree model through Optuna tuning

For a noisy pregame prediction problem like MLB, a **0.6292 ROC-AUC** is a strong portfolio result and demonstrates solid applied machine learning workflow design.